In [35]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, udf
from pyspark.sql.types import StringType, IntegerType
import time
from pyspark import StorageLevel

In [36]:
spark = SparkSession.builder.appName("advSaprk").getOrCreate()

In [3]:
df= spark.range(1,1000000).withColumn("value",col("id")%1000)

In [4]:
df.take(10)

[Row(id=1, value=1),
 Row(id=2, value=2),
 Row(id=3, value=3),
 Row(id=4, value=4),
 Row(id=5, value=5),
 Row(id=6, value=6),
 Row(id=7, value=7),
 Row(id=8, value=8),
 Row(id=9, value=9),
 Row(id=10, value=10)]

In [5]:
print("Before partitioning:",df.rdd.getNumPartitions())

Before partitioning: 2


In [6]:
df_repartitioned = df.repartition(20)
print("After partitioning:",df_repartitioned.rdd.getNumPartitions())

[Stage 1:>                                                          (0 + 2) / 2]

After partitioning: 20


In [7]:
df.write.mode("overwrite").csv("output/spark_data1",header=True)

In [8]:
df_repartitioned.write.mode("overwrite").csv("output/spark_data",header=True)

26/06/13 06:28:35 WARN GarbageCollectionMetrics: To enable non-built-in garbage collector(s) List(G1 Concurrent GC), users should configure it(them) to spark.eventLog.gcMetrics.youngGenerationGarbageCollectors or spark.eventLog.gcMetrics.oldGenerationGarbageCollectors
                                                                                

In [9]:
df_coalesced = df_repartitioned.coalesce(5)
print("After Coalesce:",df_coalesced.rdd.getNumPartitions())

[Stage 6:>                                                          (0 + 2) / 2]

After Coalesce: 5


In [10]:
df_coalesced.write.mode("overwrite").csv("output/spark_data2",header=True)

In [11]:
optimized_df=df.filter(col("value")>500).filter(col("id")<50000).select("id","value")
optimized_df.explain()

== Physical Plan ==
*(1) Project [id#0L, (id#0L % 1000) AS value#2L]
+- *(1) Filter (((id#0L % 1000) > 500) AND (id#0L < 50000))
   +- *(1) Range (1, 1000000, step=1, splits=2)




In [12]:
start_time=time.time()
count_uncached = optimized_df.count()
end_time= time.time()
print(f"1.Optimized execution | Count: {count_uncached} | Time : {round(end_time - start_time, 4)} seconds")

1.Optimized execution | Count: 24950 | Time : 0.3876 seconds


In [13]:
optimized_df.cache()

DataFrame[id: bigint, value: bigint]

In [14]:
start_time=time.time()
count_uncached = optimized_df.count()
end_time= time.time()
print(f"1.first execution | Count: {count_uncached} | Time : {round(end_time - start_time, 4)} seconds")

1.first execution | Count: 24950 | Time : 0.7751 seconds


In [15]:
start_time=time.time()
count_uncached = optimized_df.count()
end_time= time.time()
print(f"1.Second execution | Count: {count_uncached} | Time : {round(end_time - start_time, 4)} seconds")

1.Second execution | Count: 24950 | Time : 0.2123 seconds


In [16]:
optimized_df.unpersist()

DataFrame[id: bigint, value: bigint]

In [17]:
start_time=time.time()
count_uncached = optimized_df.count()
end_time= time.time()
print(f"1.Optimized execution | Count: {count_uncached} | Time : {round(end_time - start_time, 4)} seconds")

1.Optimized execution | Count: 24950 | Time : 0.1581 seconds


In [23]:
##
data=[("Alice",35),("Bob",16),("Charlie",16),("Mansi",25)]
df=spark.createDataFrame(data,["Name","Age"])
df.show()

+-------+---+
|   Name|Age|
+-------+---+
|  Alice| 35|
|    Bob| 16|
|Charlie| 16|
|  Mansi| 25|
+-------+---+



In [24]:
# def categorize_age(age):
#     if age>=18:
#         return "Adult"
#     return "Minor"

In [41]:
def can_drive(age):
    if(age>18):
        return "Can drive"
    elif(age>13 and age <18):
        return "Can drive with learning license"
    return "Cannot drive"

In [42]:
can_drive_udf = udf(can_drive, StringType())

In [43]:
df_method1= df.withColumn("Permission",can_drive_udf(col("Age")))
df_method1.show()

+-------+---+--------------------+
|   Name|Age|          Permission|
+-------+---+--------------------+
|  Alice| 35|           Can drive|
|    Bob| 16|Can drive with le...|
|Charlie| 16|Can drive with le...|
|  Mansi| 25|           Can drive|
+-------+---+--------------------+



In [44]:
spark.udf.register("sql_can_drive",can_drive, StringType())
df.createOrReplaceTempView("People")

In [45]:
sql_df = spark.sql('''
SELECT 
  Name,
  Age,
  sql_can_drive(age) AS Permission
FROM People
''')

In [46]:
sql_df.show()

+-------+---+--------------------+
|   Name|Age|          Permission|
+-------+---+--------------------+
|  Alice| 35|           Can drive|
|    Bob| 16|Can drive with le...|
|Charlie| 16|Can drive with le...|
|  Mansi| 25|           Can drive|
+-------+---+--------------------+



In [47]:
spark.stop()